In [ ]:
import torch
from datasets import load_dataset
import numpy as np
import pandas as pd
from allennlp.modules.scalar_mix import ScalarMix
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import shuffle
from tqdm import tqdm
import transformers

In [ ]:
device = torch.device('cuda')

#Original dataset takes too long to load because of images
ds = load_dataset("tasksource/ScienceQA_text_only")
label_names = ['elementary', 'middle', 'high']

def grade_to_label(grade):
    if isinstance(grade, str):
        num = int(''.join(filter(str.isdigit, grade)))
    else:
        num = int(grade)
    if 1 <= num <= 5:
        return 0  # elementary
    elif 6 <= num <= 8:
        return 1  # middle
    elif 9 <= num <= 12:
        return 2  # high
    return None

def prepare_scienceqa(split_data):
    texts = []
    labels = []
    for row in split_data:
        label = grade_to_label(row.get('grade'))
        if label is None:
            continue

        question = row['question']
        choices = row['choices']
        answer_idx = row['answer']
        answer_text = choices[answer_idx] if answer_idx < len(choices) else ""
        lecture = row.get('lecture', '') or ''
        solution = row.get('solution', '') or ''

        choices_str = " [SEP] ".join(
            [f"({chr(65+i)}) {c}" + ("(Correct Answer)" if i == answer_idx else "")
             for i, c in enumerate(choices)]
        )

        text = f"""Question: {question}
        Choices: {', '.join(f'{chr(65+i)}) {c}' for i, c in enumerate(choices))}
        Correct Answer: {chr(65+answer_idx)}) {answer_text}
        Explanation: {lecture}
        Solution: {solution}"""

        texts.append(text)
        labels.append(label)
    return np.array(texts), np.array(labels)

X_train, y_train = prepare_scienceqa(ds['train'])
X_test, y_test = prepare_scienceqa(ds['test'])

for i in range(10):
    print(f"Label: {y_train[i]} | Text: {X_train[i]}")
    print("\n")


Label: 0 | Text: Question: Which tense does the sentence use?
Mona will print her name with care.
        Choices: A) present tense, B) future tense, C) past tense
        Correct Answer: B) future tense
        Explanation: Present tense verbs tell you about something that is happening now.
Most present-tense verbs are regular. They have no ending, or they end in -s or -es.
Two verbs are irregular in the present tense, to be and to have. You must remember their forms.
Past tense verbs tell you about something that has already happened.
Most past-tense verbs are regular. They end in -ed.
Some verbs are irregular in the past tense. You must remember their past-tense forms.
Future tense verbs tell you about something that is going to happen.
All future-tense verbs use the word will.
Present | Past | Future
walk, walks | walked | will walk
go, goes | went | will go
        Solution: The sentence is in future tense. You can tell because it uses will before the main verb, print. The verb te

In [ ]:
class ScoringModel(torch.nn.Module):
    def __init__(self, language_model, prefix, num_classes=3) -> None:
        super().__init__()
        self.prefix = prefix

        self.tokenizer = AutoTokenizer.from_pretrained(language_model)
        self.lm = AutoModel.from_pretrained(language_model).to(device)
        self.scalar_mix = ScalarMix(self.lm.config.num_hidden_layers + 1)

        self.dropout = torch.nn.Dropout(p=0.2)

        self.lm_name = language_model

        self.classification_head = torch.nn.Sequential(
            torch.nn.Linear(self.lm.config.hidden_size, self.lm.config.hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(self.lm.config.hidden_size, num_classes)
        )

        self.loss = torch.nn.CrossEntropyLoss()

        self.X = None
        self.y = None
        self.eval_X = None
        self.eval_y = None

    def set_dataset(self, X, y):
        self.X = X
        self.y = y

    def set_evalset(self, X, y):
        self.eval_X = X
        self.eval_y = y

    def self_eval(self):
        #Todo
        return None

    def forward(self, input):
        inputs = self.tokenizer(input, return_tensors='pt', padding=True, truncation=True, max_length=self.lm.config.max_position_embeddings - 2)
        outputs = self.lm(**inputs.to(device), output_hidden_states=True)
        hidden_states = outputs.hidden_states

        result = self.classification_head(
            torch.mean(
                self.dropout(
                    self.scalar_mix(
                        hidden_states
                    )
                ),
                dim=1
            )
        )
        return result

    def fit(
        self,
        epochs,
        optimizer,
        scheduler,
        batch_size=4
    ) -> None:
        self.train()
        for epoch in range(epochs):
            print(f"Epoch {epoch}")
            r = 0.0
            num_s = 0.0
            d, d_y = shuffle(self.X, self.y)

            batches_X = [
                d[n:n+batch_size] for n in range(0, len(d), batch_size)
            ]
            batches_y = [
                d_y[n:n+batch_size] for n in range(0, len(d_y), batch_size)
            ]

            for batch in tqdm(range(len(batches_X))):
                pred = self.forward(list(batches_X[batch]))
                ls = self.loss(pred, batches_y[batch])

                optimizer.zero_grad()
                ls.backward()
                optimizer.step()
                scheduler.step()

                r += ls.detach().item()
                num_s += 1

                if batch % 10 == 0 and batch > 0:
                    print(str(r / num_s))

            if not (self.eval_X is None):
                ev = self.self_eval()
                print(ev)
                self.train()
